In [0]:
from pyspark.sql import functions as F

raw_df = spark.read.table("gh_archive_lakehouse_dev.default.raw_gh_archive") 


In [0]:
flat_df = raw_df.select(
    F.col("id"),
    F.col("type"),
    F.to_timestamp(F.col("created_at")).alias("created_at"),
    F.to_date(F.col("created_at")).alias("event_date"),
    F.col("actor.avatar_url").alias("actor_avatar_url"),
    F.col("actor.display_login").alias("actor_display_login"),
    F.col("actor.gravatar_id").alias("actor_gravatar_id"),
    F.col("actor.id").alias("actor_id"),
    F.col("actor.login").alias("actor_login"),
    F.col("actor.url").alias("actor_url"),
    F.col("org.avatar_url").alias("org_avatar_url"),
    F.col("org.gravatar_id").alias("org_gravatar_id"),
    F.col("org.id").alias("org_id"),
    F.col("org.login").alias("org_login"),
    F.col("org.url").alias("org_url"),
    F.col("repo.id").alias("repo_id"),
    F.col("repo.name").alias("repo_name"),
    F.col("repo.url").alias("repo_url"),
    F.col("payload").alias("payload")
)

In [0]:
def row_count_sanity_check(raw_df, flat_df):
  return flat_df.count() == raw_df.count()

In [0]:
def check_nulls(flat_df, col):
    return flat_df.filter(F.col(col).isNull()).limit(1).count() == 0

In [0]:
def is_unique(flat_df):
    return not(flat_df.count() > flat_df.select("id").distinct().count())

In [0]:
def quality_check(raw_df, flat_df, required_cols=("id", "actor_id", "type", "created_at")):
    results = {
        "row_count_sanity": row_count_sanity_check(raw_df, flat_df),
        "id_unique": is_unique(flat_df)
    }

    for col in required_cols:
        results[f"{col}_not_null"] = check_nulls(flat_df, col)

    results["all_passed"] = all(results.values())
    return results


In [0]:
def upsert_with_quality_check(dest_table, raw_df, flat_df, full_refresh=False):
    qc_results = quality_check(raw_df, flat_df)
    failed_checks = [name for name, passed in qc_results.items() if name != "all_passed" and not passed]

    if not qc_results["all_passed"]:
        raise ValueError(f"Quality check failed: {failed_checks}")

    if not spark.catalog.tableExists(dest_table) or full_refresh:
        (
            flat_df
            .write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(dest_table)
        )
    else:
        (
            flat_df
            .write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(dest_table)
        )

dest_table = "default.stg_gh_archive"
upsert_with_quality_check(dest_table, raw_df, flat_df)
